1. Introduction  
Objective: select best production model  
Decision criteria:  
ROC-AUC  
Recall  
Interpretability  
Inference speed  

2. Load Processed Data  
Same dataset as notebook 02 (no re-cleaning).  

3. Load Baseline Results  
Either:  
Load CSV exported from notebook 02  
OR  
Manually recreate summary table  
This ensures fair comparison.  

4. Ensemble Models Implementation  
Models to test:  
XGBoost  
LightGBM  
AdaBoost  
Stacking Ensemble  
Each model:  
Pipeline  
Imbalance handling  
Hyperparameter tuning (Optuna or GridSearch)  

5. Cross-Validation Results
Stratified 5-fold CV  
Mean ± std ROC-AUC  
Mean Recall  
Plot:  
Boxplot of ROC-AUC across folds  

6. Learning Curves  
For top 2–3 models:  
Training vs validation score  
Detect overfitting  
Saved to /figures  

7. Validation Curves  
One key hyperparameter per model  
Show sensitivity  

8. Feature Importance  
Depending on model:  
Tree-based importance  
SHAP values (preferred)  
Top 10 features  

9. Model Interpretability  
SHAP summary plot  
SHAP force plot (example patient)  
OR  
LIME explanation  
Explain clinical relevance.  

10. Final Model Selection  
Markdown justification:  
Why this model  
Trade-offs  
Comparison vs baseline  

---
Define the same dataset use for the best ROC_AUC model from "02_ML_Models.ipynb"

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    
from src.models.traditional_ml import (
    load_dataset
)

DATA_DIR = '../data/processed'
FIG_DIR = '../figures/02_ML_Models'
MODELS_DIR = '../models'
RANDOM_STATE = 42

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

comparison_traditional = 'model_comparison.csv'
traditional_comparison_df = load_dataset(os.path.join(DATA_DIR, comparison_traditional))

display(traditional_comparison_df.sort_values("test_roc_auc", ascending=False).head())

,dataset_version,model_name,imbalance_method,best_params,cv_best_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,standard_scaled,svm,class_weight,"{""model__C"": 5, ""model__gamma"": ""scale"", ""mode...",0.721043,0.669811,0.252459,0.596899,0.354839,0.689712
1,unscaled,svm,class_weight,"{""model__C"": 5, ""model__gamma"": ""scale"", ""mode...",0.721074,0.668632,0.251634,0.596899,0.354023,0.689712
2,robust_scaled,svm,class_weight,"{""model__C"": 5, ""model__gamma"": ""scale"", ""mode...",0.720986,0.668632,0.251634,0.596899,0.354023,0.689588
7,robust_scaled,svm,smote,"{""model__C"": 5, ""model__gamma"": ""scale"", ""mode...",0.720929,0.650943,0.244648,0.620155,0.350877,0.688887
8,robust_scaled,svm,smote+class_weight,"{""model__C"": 5, ""model__gamma"": ""scale"", ""mode...",0.720929,0.650943,0.244648,0.620155,0.350877,0.688887


In [10]:
best_row = traditional_comparison_df.iloc[0]
best_traditional = f"{best_row.dataset_version}"
train_filepath = os.path.join(DATA_DIR, f'train_{best_traditional}.csv')
test_filepath = os.path.join(DATA_DIR, f'test_{best_traditional}.csv')
train_df = load_dataset(train_filepath)
test_df = load_dataset(test_filepath)

print(f"The data used in the best traditional model is imported for comparison: {best_traditional}.")

The data used in the best traditional model is imported for comparison: unscaled.


In [14]:
"""
Advanced Models & Model Selection - Collaborator 5
Coronary Heart Disease Prediction Project
Timeline: Jan 1-9, 2025
Branch: feature/advanced-models

Final Version: Uses scale_pos_weight and Optuna tuning
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

# Model libraries
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_validate,
    learning_curve,
    validation_curve
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve
)

# Ensemble models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler

# Interpretability
import shap

# =============================================================================
# CONFIGURATION
# =============================================================================
RANDOM_STATE = 42
TARGET_COL = 'TenYearCHD'

# =============================================================================
# 1. DATA LOADING AND PREPARATION
# =============================================================================

def load_presplit_data(train_filepath=train_filepath,
                       test_filepath=test_filepath,
                       target_col_name='TenYearCHD'):
    """
    Load pre-split train and test datasets
    """
    print("="*80)
    print("STEP 1: LOADING PRE-SPLIT DATA")
    print("="*80)
    
    # Load training data
    print(f"\n✓ Loading training data from: {train_filepath}")
    train_df = pd.read_csv(train_filepath)
    print(f"  Shape: {train_df.shape}")
    
    # Load test data
    print(f"\n✓ Loading test data from: {test_filepath}")
    test_df = pd.read_csv(test_filepath)
    print(f"  Shape: {test_df.shape}")
    
    # Find target column
    if target_col_name not in train_df.columns:
        possible_targets = ['TenYearCHD', 'target', 'ten_year_chd']
        target_found = False
        for col in possible_targets:
            if col in train_df.columns:
                target_col_name = col
                target_found = True
                print(f"\n✓ Target column found: {target_col_name}")
                break
        
        if not target_found:
            raise ValueError(f"Target column '{target_col_name}' not found. Available columns: {train_df.columns.tolist()}")
    else:
        print(f"\n✓ Target column: {target_col_name}")
    
    # Separate features and target
    X_train = train_df.drop(columns=[target_col_name])
    y_train = train_df[target_col_name]
    
    X_test = test_df.drop(columns=[target_col_name])
    y_test = test_df[target_col_name]
    
    print(f"\n✓ Data split information:")
    print(f"  Training set: {X_train.shape}")
    print(f"  Test set: {X_test.shape}")
    
    # Verify feature consistency
    if list(X_train.columns) != list(X_test.columns):
        raise ValueError("Training and test sets have different features!")
    
    print(f"\n✓ Feature consistency verified")
    
    # Class distribution
    train_dist = y_train.value_counts().sort_index()
    print(f"\n✓ Training Set Class Distribution:")
    print(f"  Class 0 (No CHD): {train_dist[0]} ({train_dist[0]/len(y_train)*100:.2f}%)")
    print(f"  Class 1 (CHD): {train_dist[1]} ({train_dist[1]/len(y_train)*100:.2f}%)")
    print(f"  Imbalance Ratio: {train_dist[0]/train_dist[1]:.2f}:1")
    
    test_dist = y_test.value_counts().sort_index()
    print(f"\n✓ Test Set Class Distribution:")
    print(f"  Class 0 (No CHD): {test_dist[0]} ({test_dist[0]/len(y_test)*100:.2f}%)")
    print(f"  Class 1 (CHD): {test_dist[1]} ({test_dist[1]/len(y_test)*100:.2f}%)")
    print(f"  Imbalance Ratio: {test_dist[0]/test_dist[1]:.2f}:1")
    
    # Handle missing values
    train_missing = X_train.isnull().sum()
    test_missing = X_test.isnull().sum()
    
    if train_missing.any() or test_missing.any():
        print(f"\n⚠ Warning: Missing values detected")
        for col in X_train.columns:
            if X_train[col].dtype in ['float64', 'int64']:
                median_val = X_train[col].median()
                X_train[col].fillna(median_val, inplace=True)
                X_test[col].fillna(median_val, inplace=True)
            else:
                mode_val = X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else X_train[col].iloc[0]
                X_train[col].fillna(mode_val, inplace=True)
                X_test[col].fillna(mode_val, inplace=True)
    
    # Handle categorical variables
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if categorical_cols:
        print(f"\n✓ Encoding categorical variables: {categorical_cols}")
        label_encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col].astype(str))
            X_test[col] = le.transform(X_test[col].astype(str))
            label_encoders[col] = le
    else:
        label_encoders = None
        print(f"\n✓ No categorical variables found (all numeric)")
    
    feature_names = X_train.columns.tolist()
    print(f"\n✓ Features ({len(feature_names)}):")
    for i, feat in enumerate(feature_names, 1):
        print(f"  {i:2d}. {feat}")
    
    return X_train, X_test, y_train, y_test, feature_names, label_encoders


# =============================================================================
# 2. HYPERPARAMETER TUNING WITH OPTUNA
# =============================================================================

def tune_xgboost(X_train, y_train, n_trials=50):
    """
    Optimize XGBoost hyperparameters using Optuna
    """
    print("\n" + "="*80)
    print("HYPERPARAMETER TUNING - XGBoost")
    print("="*80)
    
    # Calculate scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    print(f"\n✓ Calculated scale_pos_weight: {scale_pos_weight:.2f}")
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 600),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'scale_pos_weight': scale_pos_weight,
            'eval_metric': 'auc',
            'random_state': RANDOM_STATE,
            'use_label_encoder': False
        }
        
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        
        scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model.fit(X_tr, y_tr)
            y_pred_proba = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, y_pred_proba)
            scores.append(score)
        
        return np.mean(scores)
    
    print(f"\n✓ Starting Optuna optimization ({n_trials} trials)...")
    print("  Objective: Maximize ROC-AUC")
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"\n✓ Optimization completed!")
    print(f"  Best ROC-AUC: {study.best_value:.4f}")
    print(f"  Best parameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    
    best_params = study.best_params.copy()
    best_params['scale_pos_weight'] = scale_pos_weight
    best_params['eval_metric'] = 'auc'
    best_params['random_state'] = RANDOM_STATE
    best_params['use_label_encoder'] = False
    
    return best_params


def tune_lightgbm(X_train, y_train, n_trials=50):
    """
    Optimize LightGBM hyperparameters using Optuna
    """
    print("\n" + "="*80)
    print("HYPERPARAMETER TUNING - LightGBM")
    print("="*80)
    
    # Calculate scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    print(f"\n✓ Calculated scale_pos_weight: {scale_pos_weight:.2f}")
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 600),
            'num_leaves': trial.suggest_int('num_leaves', 20, 50),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'scale_pos_weight': scale_pos_weight,
            'random_state': RANDOM_STATE,
            'verbose': -1
        }
        
        model = LGBMClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        
        scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model.fit(X_tr, y_tr)
            y_pred_proba = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, y_pred_proba)
            scores.append(score)
        
        return np.mean(scores)
    
    print(f"\n✓ Starting Optuna optimization ({n_trials} trials)...")
    print("  Objective: Maximize ROC-AUC")
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"\n✓ Optimization completed!")
    print(f"  Best ROC-AUC: {study.best_value:.4f}")
    print(f"  Best parameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    
    best_params = study.best_params.copy()
    best_params['scale_pos_weight'] = scale_pos_weight
    best_params['random_state'] = RANDOM_STATE
    best_params['verbose'] = -1
    
    return best_params


def tune_adaboost(X_train, y_train, n_trials=40):
    """
    Optimize AdaBoost hyperparameters using Optuna
    Uses sample_weight to handle imbalance
    """
    print("\n" + "="*80)
    print("HYPERPARAMETER TUNING - AdaBoost")
    print("="*80)

    # Calculate sample weights for imbalance
    class_counts = y_train.value_counts()
    class_weight = len(y_train) / (2 * class_counts)
    sample_weights = y_train.map(class_weight)
    
    print(f"\n✓ Using sample_weight to handle imbalance")
    print(f"  Class 0 weight: {class_weight[0]:.3f}")
    print(f"  Class 1 weight: {class_weight[1]:.3f}")
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 400),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
            'algorithm': 'SAMME',  # Better for probability estimates
            'random_state': RANDOM_STATE,
        }
        
        model = AdaBoostClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        
        scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            sw_tr = sample_weights.iloc[train_idx]
            
            # Fit with sample weights
            model.fit(X_tr, y_tr, sample_weight=sw_tr)
            y_pred_proba = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, y_pred_proba)
            scores.append(score)
        
        return np.mean(scores)
    
    print(f"\n✓ Starting Optuna optimization ({n_trials} trials)...")
    print("  Objective: Maximize ROC-AUC")
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"\n✓ Optimization completed!")
    print(f"  Best ROC-AUC: {study.best_value:.4f}")
    print(f"  Best parameters:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    
    best_params = study.best_params.copy()
    best_params['random_state'] = RANDOM_STATE
    
    return best_params


# =============================================================================
# 3. MODEL TRAINING AND EVALUATION
# =============================================================================

def train_base_models(X_train, y_train, X_test, y_test, 
                     best_xgb_params, best_lgbm_params, best_ada_params):
    """
    Train base ensemble models with optimized hyperparameters
    """
    print("\n" + "="*80)
    print("TRAINING BASE ENSEMBLE MODELS")
    print("="*80)
    
    models = {
        'XGBoost': XGBClassifier(**best_xgb_params),
        'LightGBM': LGBMClassifier(**best_lgbm_params),
        'AdaBoost': AdaBoostClassifier(**best_ada_params)
    }
    
    results = {}
    trained_models = {}
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }
    
    for name, model in models.items():
        print(f"\n{'='*80}")
        print(f"Training: {name}")
        print(f"{'='*80}")
        
        # Cross-validation
        print(f"  Running 5-fold cross-validation...")

        if name == 'AdaBoost':
            # For AdaBoost, we custom CV with sample_weight
            cv_scores = {'test_roc_auc': [], 'test_recall': [], 'test_precision': [], 
                        'test_f1': [], 'test_accuracy': []}
            
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

                # Calculate sample weights for imbalance
            class_counts = y_train.value_counts()
            class_weight = len(y_train) / (2 * class_counts)
            sample_weights = y_train.map(class_weight)
            for train_idx, val_idx in cv.split(X_train, y_train):
                X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                sw_tr = sample_weights.iloc[train_idx]
                
                model.fit(X_tr, y_tr, sample_weight=sw_tr)
                y_pred = model.predict(X_val)
                y_proba = model.predict_proba(X_val)[:, 1]
                
                cv_scores['test_roc_auc'].append(roc_auc_score(y_val, y_proba))
                cv_scores['test_recall'].append(recall_score(y_val, y_pred))
                cv_scores['test_precision'].append(precision_score(y_val, y_pred, zero_division=0))
                cv_scores['test_f1'].append(f1_score(y_val, y_pred))
                cv_scores['test_accuracy'].append(accuracy_score(y_val, y_pred))
            
            cv_results = {k: np.array(v) for k, v in cv_scores.items()}
            
            # Train final model with sample weights
            model.fit(X_train, y_train, sample_weight=sample_weights)
        else:
            cv_results = cross_validate(
                model, X_train, y_train,
                cv=cv, scoring=scoring, n_jobs=-1,
                return_train_score=True
            )
            model.fit(X_train, y_train)
        
        # Predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        y_test_proba = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        train_metrics = {
            'accuracy': accuracy_score(y_train, y_train_pred),
            'precision': precision_score(y_train, y_train_pred, zero_division=0),
            'recall': recall_score(y_train, y_train_pred),
            'f1': f1_score(y_train, y_train_pred),
            'roc_auc': roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        }
        
        test_metrics = {
            'accuracy': accuracy_score(y_test, y_test_pred),
            'precision': precision_score(y_test, y_test_pred, zero_division=0),
            'recall': recall_score(y_test, y_test_pred),
            'f1': f1_score(y_test, y_test_pred),
            'roc_auc': roc_auc_score(y_test, y_test_proba)
        }
        
        cv_metrics = {
            'cv_accuracy_mean': cv_results['test_accuracy'].mean(),
            'cv_accuracy_std': cv_results['test_accuracy'].std(),
            'cv_precision_mean': cv_results['test_precision'].mean(),
            'cv_precision_std': cv_results['test_precision'].std(),
            'cv_recall_mean': cv_results['test_recall'].mean(),
            'cv_recall_std': cv_results['test_recall'].std(),
            'cv_f1_mean': cv_results['test_f1'].mean(),
            'cv_f1_std': cv_results['test_f1'].std(),
            'cv_roc_auc_mean': cv_results['test_roc_auc'].mean(),
            'cv_roc_auc_std': cv_results['test_roc_auc'].std(),
        }
        
        # Store results
        results[name] = {
            'train_metrics': train_metrics,
            'metrics': test_metrics,
            'cv_metrics': cv_metrics,
            'confusion_matrix': confusion_matrix(y_test, y_test_pred),
            'y_pred': y_test_pred,
            'y_proba': y_test_proba,
            'classification_report': classification_report(y_test, y_test_pred, output_dict=True)
        }
        
        trained_models[name] = model
        
        # Print results
        print(f"\n  Cross-Validation Results (mean ± std):")
        print(f"    Accuracy:  {cv_metrics['cv_accuracy_mean']:.4f} ± {cv_metrics['cv_accuracy_std']:.4f}")
        print(f"    Precision: {cv_metrics['cv_precision_mean']:.4f} ± {cv_metrics['cv_precision_std']:.4f}")
        print(f"    Recall:    {cv_metrics['cv_recall_mean']:.4f} ± {cv_metrics['cv_recall_std']:.4f}")
        print(f"    F1-Score:  {cv_metrics['cv_f1_mean']:.4f} ± {cv_metrics['cv_f1_std']:.4f}")
        print(f"    ROC-AUC:   {cv_metrics['cv_roc_auc_mean']:.4f} ± {cv_metrics['cv_roc_auc_std']:.4f}")
        
        print(f"\n  Test Set Performance:")
        print(f"    Accuracy:  {test_metrics['accuracy']:.4f}")
        print(f"    Precision: {test_metrics['precision']:.4f}")
        print(f"    Recall:    {test_metrics['recall']:.4f}")
        print(f"    F1-Score:  {test_metrics['f1']:.4f}")
        print(f"    ROC-AUC:   {test_metrics['roc_auc']:.4f}")
    
    return results, trained_models


def create_stacking_ensemble(base_models, X_train, y_train):
    """
    Create stacking ensemble with Logistic Regression meta-learner
    """
    print("\n" + "="*80)
    print("CREATING STACKING ENSEMBLE")
    print("="*80)
    
    estimators = [
        ('lr', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_STATE)),
        ('xgb', base_models['XGBoost']),
        ('lgbm', base_models['LightGBM'])
    ]
    
    stacking_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_STATE),
        cv=5,
        n_jobs=-1
    )
    
    print("\n✓ Training stacking ensemble...")
    print(f"  Base estimators: {len(estimators)}")
    print(f"  Meta-learner: Logistic Regression")
    
    stacking_clf.fit(X_train, y_train)
    
    print("✓ Stacking ensemble trained successfully!")
    
    return stacking_clf


def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive model evaluation
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    
    return {
        'metrics': metrics,
        'confusion_matrix': cm,
        'classification_report': report,
        'y_pred': y_pred,
        'y_proba': y_proba
    }


# =============================================================================
# 4. THRESHOLD OPTIMIZATION
# =============================================================================

def optimize_threshold_for_recall(model, X, y):
    """
    Optimize classification threshold to maximize recall
    """
    probs = model.predict_proba(X)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y, probs)
    
    # Find threshold that maximizes recall
    idx = np.argmax(recall)
    optimal_threshold = thresholds[idx] if idx < len(thresholds) else 0.5
    optimal_recall = recall[idx]
    
    return optimal_threshold, optimal_recall


# =============================================================================
# 5. VISUALIZATION
# =============================================================================

def plot_roc_curves(results_dict, y_test, save_path='roc_curves_comparison.png'):
    """
    Plot ROC curves for all models
    """
    plt.figure(figsize=(10, 8))
    
    for name, results in results_dict.items():
        fpr, tpr, _ = roc_curve(y_test, results['y_proba'])
        auc = results['metrics']['roc_auc']
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", linewidth=2)
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5000)')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ ROC curves saved: {save_path}")
    plt.close()


def plot_confusion_matrices(results_dict, save_path='confusion_matrices.png'):
    """
    Plot confusion matrices for all models
    """
    n_models = len(results_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
    
    if n_models == 1:
        axes = [axes]
    
    for idx, (name, results) in enumerate(results_dict.items()):
        cm = results['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=['No CHD', 'CHD'],
                   yticklabels=['No CHD', 'CHD'],
                   ax=axes[idx])
        axes[idx].set_title(f'{name}', fontweight='bold')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Confusion matrices saved: {save_path}")
    plt.close()


# =============================================================================
# 6. FEATURE IMPORTANCE AND INTERPRETABILITY
# =============================================================================

def analyze_feature_importance(model, feature_names, model_name='Model'):
    """
    Analyze and plot feature importance
    """
    print(f"\n{'='*80}")
    print(f"Feature Importance Analysis - {model_name}")
    print(f"{'='*80}")
    
    if not hasattr(model, 'feature_importances_'):
        print(f"  {model_name} does not have feature_importances_ attribute")
        return None
    
    importance = model.feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    print(f"\nTop 10 Most Important Features:")
    for idx, row in importance_df.head(10).iterrows():
        print(f"  {row['feature']:25s}: {row['importance']:.4f}")
    
    # Plot
    plt.figure(figsize=(10, 8))
    top_n = min(15, len(importance_df))
    plt.barh(range(top_n), importance_df['importance'].head(top_n))
    plt.yticks(range(top_n), importance_df['feature'].head(top_n))
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Feature Importance - {model_name}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    filename = f'feature_importance_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Feature importance plot saved: {filename}")
    plt.close()
    
    return importance_df


def shap_analysis(model, X_test, feature_names, model_name='Model', max_display=15):
    """
    SHAP analysis for model interpretability
    """
    print(f"\n{'='*80}")
    print(f"SHAP Analysis - {model_name}")
    print(f"{'='*80}")
    
    try:
        print("\n✓ Creating SHAP explainer...")
        explainer = shap.TreeExplainer(model)
        
        sample_size = min(500, len(X_test))
        X_sample = X_test.sample(n=sample_size, random_state=RANDOM_STATE)
        
        print(f"✓ Calculating SHAP values for {sample_size} samples...")
        shap_values = explainer.shap_values(X_sample)
        
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        
        # Summary plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                         max_display=max_display, show=False)
        plt.tight_layout()
        filename = f'shap_summary_{model_name.lower().replace(" ", "_")}.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✓ SHAP summary plot saved: {filename}")
        plt.close()
        
        shap_importance = np.abs(shap_values).mean(axis=0)
        shap_df = pd.DataFrame({
            'feature': feature_names,
            'shap_importance': shap_importance
        }).sort_values('shap_importance', ascending=False)
        
        print(f"\nTop 10 Features by SHAP Importance:")
        for idx, row in shap_df.head(10).iterrows():
            print(f"  {row['feature']:25s}: {row['shap_importance']:.4f}")
        
        return shap_df
        
    except Exception as e:
        print(f"✗ SHAP analysis failed: {str(e)}")
        return None


# =============================================================================
# 7. LEARNING AND VALIDATION CURVES
# =============================================================================

def plot_learning_curves(model, X_train, y_train, model_name='Model'):
    """
    Plot learning curves
    """
    print(f"\n{'='*80}")
    print(f"Learning Curves - {model_name}")
    print(f"{'='*80}")
    
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=np.linspace(0.1, 1.0, 5),
        cv=5,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                     alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, 'o-', color='red', label='Cross-validation score')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                     alpha=0.1, color='red')
    
    plt.xlabel('Training Set Size', fontsize=12)
    plt.ylabel('ROC-AUC Score', fontsize=12)
    plt.title(f'Learning Curves - {model_name}', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    filename = f'learning_curves_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Learning curves saved: {filename}")
    plt.close()
    
    return {
        'train_sizes': train_sizes.tolist(),
        'train_auc': train_mean.tolist(),
        'val_auc': val_mean.tolist()
    }


def plot_validation_curve(model, X_train, y_train, model_name='Model'):
    """
    Plot validation curve for n_estimators
    """
    print(f"\n{'='*80}")
    print(f"Validation Curve - {model_name}")
    print(f"{'='*80}")
    
    if not hasattr(model, 'n_estimators'):
        print(f"  Model does not have n_estimators parameter")
        return None
    
    param_range = [50, 100, 200, 400]
    
    train_scores, val_scores = validation_curve(
        model, X_train, y_train,
        param_name='n_estimators',
        param_range=param_range,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(param_range, train_mean, 'o-', color='blue', label='Training score')
    plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, 
                     alpha=0.1, color='blue')
    plt.plot(param_range, val_mean, 'o-', color='red', label='Cross-validation score')
    plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, 
                     alpha=0.1, color='red')
    
    plt.xlabel('n_estimators', fontsize=12)
    plt.ylabel('ROC-AUC Score', fontsize=12)
    plt.title(f'Validation Curve - {model_name} - n_estimators', 
              fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    filename = f'validation_curve_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Validation curve saved: {filename}")
    plt.close()
    
    return {
        'param_name': 'n_estimators',
        'param_range': param_range,
        'train_auc': train_mean.tolist(),
        'val_auc': val_mean.tolist()
    }


# =============================================================================
# 8. MODEL SELECTION
# =============================================================================

def select_best_model(results_dict, models_dict, X_test, y_test):
    """
    Select best model based on weighted score
    Weight: 0.7 * ROC-AUC + 0.3 * Recall
    """
    print("\n" + "="*80)
    print("MODEL SELECTION")
    print("="*80)
    
    final_scores = {}
    for name, model in models_dict.items():
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        
        roc_auc = roc_auc_score(y_test, y_proba)
        recall = recall_score(y_test, y_pred)
        
        final_scores[name] = 0.7 * roc_auc + 0.3 * recall
    
    best_model_name = max(final_scores, key=final_scores.get)
    best_model = models_dict[best_model_name]
    
    print("\n✓ Final Weighted Scores (0.7*ROC_AUC + 0.3*Recall):")
    for name, score in sorted(final_scores.items(), key=lambda x: x[1], reverse=True):
        print(f"  {name:15s}: {score:.4f}")
    
    print("\n" + "="*80)
    print(f"🏆 BEST MODEL: {best_model_name}")
    print(f"   Weighted Score: {final_scores[best_model_name]:.4f}")
    print("="*80)
    
    return best_model_name, best_model, final_scores


# =============================================================================
# 9. SAVE MODEL AND METADATA
# =============================================================================

def save_final_model(model, model_name, feature_names, metrics, 
                     hyperparameters, final_scores, optimal_threshold,
                     optimal_recall, learning_curve_data, validation_curve_data,
                     label_encoders=None, save_dir='models'):
    """
    Save the final selected model with complete metadata
    """
    print("\n" + "="*80)
    print("SAVING FINAL MODEL")
    print("="*80)
    
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # Save model
    model_path = os.path.join(save_dir, 'best_model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"\n✓ Model saved: {model_path}")
    
    # Save metadata
    metadata = {
        'model_name': model_name,
        'model_type': type(model).__name__,
        'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'feature_names': feature_names,
        'n_features': len(feature_names),
        'test_metrics': metrics,
        'tuned_hyperparameters': hyperparameters,
        'final_scores': final_scores,
        'optimal_threshold': float(optimal_threshold),
        'optimal_recall': float(optimal_recall),
        'learning_curve': learning_curve_data,
        'validation_curve': validation_curve_data,
        'imbalance_handling': 'scale_pos_weight',
        'hyperparameter_tuning': 'Optuna (50 trials for boosting, 40 for AdaBoost)',
        'label_encoders': {k: v.classes_.tolist() for k, v in label_encoders.items()} 
                          if label_encoders else None,
    }
    
    metadata_path = os.path.join(save_dir, 'model_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"✓ Metadata saved: {metadata_path}")
    
    if label_encoders:
        encoders_path = os.path.join(save_dir, 'label_encoders.pkl')
        with open(encoders_path, 'wb') as f:
            pickle.dump(label_encoders, f)
        print(f"✓ Label encoders saved: {encoders_path}")
    
    print(f"\n✓ All artifacts saved to: {save_dir}/")
    
    return model_path, metadata_path


def create_model_comparison_csv(results_dict, output_file='model_comparison_boosting.csv'):
    """
    Create CSV file comparing all models
    """
    print("\n" + "="*80)
    print("CREATING MODEL COMPARISON TABLE")
    print("="*80)
    
    comparison_data = []
    
    for name, results in results_dict.items():
        metrics = results['metrics']
        row = {
            'Model': name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1-Score': metrics['f1'],
            'ROC-AUC': metrics['roc_auc']
        }
        comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    df = df.sort_values('ROC-AUC', ascending=False)
    df.to_csv(output_file, index=False)
    
    print(f"\n✓ Model comparison saved: {output_file}")
    print("\nModel Comparison Table:")
    print(df.to_string(index=False))
    
    return df


# =============================================================================
# 10. MAIN EXECUTION PIPELINE
# =============================================================================

def main(train_filepath=None,
         test_filepath=None,
         target_col_name='TenYearCHD'):
    """
    Main execution pipeline for Collaborator 5
    """
    print("\n" + "="*80)
    print("CORONARY HEART DISEASE PREDICTION PROJECT")
    print("Advanced Models & Model Selection - Collaborator 5")
    print("="*80)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    # =========================================================================
    # STEP 1: Load data
    # =========================================================================
    assert train_filepath and test_filepath, "Please provide train and test file paths"
    X_train, X_test, y_train, y_test, feature_names, label_encoders = load_presplit_data(
        train_filepath=train_filepath,
        test_filepath=test_filepath,
        target_col_name=target_col_name
    )
    
    # =========================================================================
    # STEP 2: Hyperparameter tuning with Optuna
    # =========================================================================
    best_xgb_params = tune_xgboost(X_train, y_train, n_trials=50)
    best_lgbm_params = tune_lightgbm(X_train, y_train, n_trials=50)
    best_ada_params = tune_adaboost(X_train, y_train, n_trials=40)
    
    all_hyperparameters = {
        'xgboost': best_xgb_params,
        'lightgbm': best_lgbm_params,
        'adaboost': best_ada_params
    }
    
    # =========================================================================
    # STEP 3: Train base models
    # =========================================================================
    base_results, base_models = train_base_models(
        X_train, y_train, X_test, y_test,
        best_xgb_params, best_lgbm_params, best_ada_params
    )
    
    # =========================================================================
    # STEP 4: Create stacking ensemble
    # =========================================================================
    stacking_model = create_stacking_ensemble(base_models, X_train, y_train)
    
    # =========================================================================
    # STEP 5: Evaluate all models
    # =========================================================================
    all_models = {
        'XGBoost': base_models['XGBoost'],
        'LightGBM': base_models['LightGBM'],
        'AdaBoost': base_models['AdaBoost'],
        'Stacking': stacking_model
    }
    
    all_results = base_results.copy()
    all_results['Stacking'] = evaluate_model(stacking_model, X_test, y_test, 'Stacking')
    
    # Add CV metrics for stacking
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_results = cross_validate(
        stacking_model, X_train, y_train,
        cv=cv, scoring={'roc_auc': 'roc_auc', 'recall': 'recall'}, n_jobs=-1
    )
    all_results['Stacking']['cv_roc_auc_mean'] = cv_results['test_roc_auc'].mean()
    all_results['Stacking']['cv_recall_mean'] = cv_results['test_recall'].mean()
    
    # =========================================================================
    # STEP 6: Visualizations
    # =========================================================================
    print("\n" + "="*80)
    print("GENERATING VISUALIZATIONS")
    print("="*80)
    
    plot_roc_curves(all_results, y_test)
    plot_confusion_matrices(all_results)
    
    # =========================================================================
    # STEP 7: Feature importance and SHAP
    # =========================================================================
    print("\n" + "="*80)
    print("FEATURE IMPORTANCE & INTERPRETABILITY")
    print("="*80)
    
    xgb_importance = analyze_feature_importance(
        base_models['XGBoost'], feature_names, 'XGBoost'
    )
    
    shap_importance = shap_analysis(
        base_models['XGBoost'], X_test, feature_names, 'XGBoost'
    )
    
    lgbm_importance = analyze_feature_importance(
        base_models['LightGBM'], feature_names, 'LightGBM'
    )

    shap_importance = shap_analysis(
        base_models['LightGBM'], X_test, feature_names, 'LightGBM'
    )
    
    ada_importance = analyze_feature_importance(
        base_models['AdaBoost'], feature_names, 'AdaBoost'
    )

    shap_importance = shap_analysis(
        base_models['AdaBoost'], X_test, feature_names, 'AdaBoost'
    )
    
    # =========================================================================
    # STEP 8: Learning and validation curves
    # =========================================================================
    print("\n" + "="*80)
    print("LEARNING AND VALIDATION CURVES")
    print("="*80)
    
    learning_curve_xgb = plot_learning_curves(
        base_models['XGBoost'], X_train, y_train, 'XGBoost'
    )
    
    validation_curve_xgb = plot_validation_curve(
        base_models['XGBoost'], X_train, y_train, 'XGBoost'
    )

    learning_curve_lgbm = plot_learning_curves(
        base_models['LightGBM'], X_train, y_train, 'LightGBM'
    )
    
    validation_curve_lgbm = plot_validation_curve(
        base_models['LightGBM'], X_train, y_train, 'LightGBM'
    )

    learning_curve_ada = plot_learning_curves(
        base_models['AdaBoost'], X_train, y_train, 'AdaBoost'
    )
    
    validation_curve_ada = plot_validation_curve(
        base_models['AdaBoost'], X_train, y_train, 'AdaBoost'
    )
    
    # =========================================================================
    # STEP 9: Create comparison table
    # =========================================================================
    comparison_df = create_model_comparison_csv(all_results)
    return comparison_df

In [ ]:
# =============================================================================
# RUN THE PIPELINE
# =============================================================================

if __name__ == "__main__":
    """
    Run the Advanced Models pipeline
    
    Usage:
    ------
    python advanced_models.py
    
    Or with custom paths:
    results = main(
        train_filepath='processed/train_unscaled.csv',
        test_filepath='processed/test_unscaled.csv'
    )
    """
    
    results = main(
        train_filepath=train_filepath,
        test_filepath=test_filepath,
        target_col_name='TenYearCHD'
    )

In [15]:
data = ["unscaled", "standard_scaled", "robust_scaled"]

for i in data:
    train_filepath = os.path.join(DATA_DIR, f'train_{i}.csv')
    test_filepath = os.path.join(DATA_DIR, f'test_{i}.csv')
    results = main(
        train_filepath=train_filepath,
        test_filepath=test_filepath,
        target_col_name='TenYearCHD'
    )

[I 2026-01-11 00:06:19,883] A new study created in memory with name: no-name-774ca7b2-97a7-4d68-ae90-82b95a6fafda



CORONARY HEART DISEASE PREDICTION PROJECT
Advanced Models & Model Selection - Collaborator 5
Start Time: 2026-01-11 00:06:19
STEP 1: LOADING PRE-SPLIT DATA

✓ Loading training data from: ../data/processed\train_unscaled.csv
  Shape: (3390, 9)

✓ Loading test data from: ../data/processed\test_unscaled.csv
  Shape: (848, 9)

✓ Target column: TenYearCHD

✓ Data split information:
  Training set: (3390, 8)
  Test set: (848, 8)

✓ Feature consistency verified

✓ Training Set Class Distribution:
  Class 0 (No CHD): 2875 (84.81%)
  Class 1 (CHD): 515 (15.19%)
  Imbalance Ratio: 5.58:1

✓ Test Set Class Distribution:
  Class 0 (No CHD): 719 (84.79%)
  Class 1 (CHD): 129 (15.21%)
  Imbalance Ratio: 5.57:1

✓ No categorical variables found (all numeric)

✓ Features (8):
   1. bmi
   2. hypertension
   3. pulse_pressure
   4. cigarettes_per_day
   5. total_cholesterol
   6. glucose
   7. heart_rate
   8. age_group_code

HYPERPARAMETER TUNING - XGBoost

✓ Calculated scale_pos_weight: 5.58

✓ Star

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-11 00:06:20,584] Trial 0 finished with value: 0.6456395103419164 and parameters: {'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.07587945476302646, 'subsample': 0.8197316968394073, 'colsample_bytree': 0.7312037280884873}. Best is trial 0 with value: 0.6456395103419164.
[I 2026-01-11 00:06:20,877] Trial 1 finished with value: 0.6734250738708316 and parameters: {'n_estimators': 262, 'max_depth': 3, 'learning_rate': 0.08795585311974417, 'subsample': 0.8202230023486418, 'colsample_bytree': 0.8416145155592091}. Best is trial 1 with value: 0.6734250738708316.
[I 2026-01-11 00:06:21,291] Trial 2 finished with value: 0.657668214436471 and parameters: {'n_estimators': 208, 'max_depth': 6, 'learning_rate': 0.08491983767203796, 'subsample': 0.7424678221356552, 'colsample_bytree': 0.73636499344142}. Best is trial 1 with value: 0.6734250738708316.
[I 2026-01-11 00:06:21,659] Trial 3 finished with value: 0.6770417897847193 and parameters: {'n_estimators': 273, 'max_depth': 4, 'le

[I 2026-01-11 00:06:40,785] A new study created in memory with name: no-name-4e71a95f-5c1b-4cde-810d-85a35be125c7


...

[I 2026-01-11 00:11:24,645] Trial 0 finished with value: 0.6509244406922752 and parameters: {'n_estimators': 350, 'num_leaves': 49, 'learning_rate': 0.07587945476302646, 'subsample': 0.8197316968394073, 'colsample_bytree': 0.7312037280884873}. Best is trial 0 with value: 0.6509244406922752.
[I 2026-01-11 00:11:24,974] Trial 1 finished with value: 0.650775854791051 and parameters: {'n_estimators': 262, 'num_leaves': 21, 'learning_rate': 0.08795585311974417, 'subsample': 0.8202230023486418, 'colsample_bytree': 0.8416145155592091}. Best is trial 0 with value: 0.6509244406922752.
[I 2026-01-11 00:11:25,554] Trial 2 finished with value: 0.6551591388771634 and parameters: {'n_estimators': 208, 'num_leaves': 50, 'learning_rate': 0.08491983767203796, 'subsample': 0.7424678221356552, 'colsample_bytree': 0.73636499344142}. Best is trial 2 with value: 0.6551591388771634.
[I 2026-01-11 00:11:26,036] Trial 3 finished with value: 0.6563241874208526 and parameters: {'n_estimators': 273, 'num_leaves':

[I 2026-01-11 00:11:53,560] A new study created in memory with name: no-name-b1a64c11-3d3a-4b36-9415-c695e60986dc


[I 2026-01-11 00:11:53,557] Trial 49 finished with value: 0.6824043900379906 and parameters: {'n_estimators': 297, 'num_leaves': 23, 'learning_rate': 0.02156103523697868, 'subsample': 0.7551793229192952, 'colsample_bytree': 0.7897151268783897}. Best is trial 46 with value: 0.6966010975094977.

✓ Optimization completed!
  Best ROC-AUC: 0.6966
  Best parameters:
    n_estimators: 262
    num_leaves: 25
    learning_rate: 0.013852246085013837
    subsample: 0.7670254305252129
    colsample_bytree: 0.7975000197829812

HYPERPARAMETER TUNING - AdaBoost

✓ Using sample_weight to handle imbalance
  Class 0 weight: 0.590
  Class 1 weight: 3.291

✓ Starting Optuna optimization (40 trials)...
  Objective: Maximize ROC-AUC


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-01-11 00:11:54,808] Trial 0 finished with value: 0.7106171380329254 and parameters: {'n_estimators': 181, 'learning_rate': 0.951207163345817}. Best is trial 0 with value: 0.7106171380329254.
[I 2026-01-11 00:11:56,869] Trial 1 finished with value: 0.7139535669058674 and parameters: {'n_estimators': 306, 'learning_rate': 0.6026718993550663}. Best is trial 1 with value: 0.7139535669058674.
[I 2026-01-11 00:11:57,554] Trial 2 finished with value: 0.7127412410299704 and parameters: {'n_estimators': 104, 'learning_rate': 0.16443457513284063}. Best is trial 1 with value: 0.7139535669058674.
[I 2026-01-11 00:11:58,004] Trial 3 finished with value: 0.7081992401857323 and parameters: {'n_estimators': 70, 'learning_rate': 0.8675143843171859}. Best is trial 1 with value: 0.7139535669058674.
[I 2026-01-11 00:11:59,821] Trial 4 finished with value: 0.7106542845082313 and parameters: {'n_estimators': 260, 'learning_rate': 0.710991852018085}. Best is trial 1 with value: 0.7139535669058674.
[I

In [4]:
comparison_boosting = "model_comparison_boosting.csv"
boosting_comparison_df = load_dataset(os.path.join(DATA_DIR, comparison_boosting))
boosting_comparison_df['Dataset_version'] ='standard_scaled'
boosting_comparison_df.loc[boosting_comparison_df['Model'] == 'XGBoost', 'Imbalance_method'] = 'scale_pos_weight'
boosting_comparison_df.loc[boosting_comparison_df['Model'] == 'LightGBM', 'Imbalance_method'] = 'scale_pos_weight'
boosting_comparison_df.loc[boosting_comparison_df['Model'] == 'AdaBoost', 'Imbalance_method'] = 'sample_weight'
boosting_comparison_df.loc[boosting_comparison_df['Model'] == 'Stacking', 'Imbalance_method'] = 'class_weight (meta-learner)'

display(boosting_comparison_df)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Dataset_version,Imbalance_method
0,Stacking,0.847877,0.000000,0.000000,0.00000,0.686203,standard_scaled,class_weight (meta-learner)
1,AdaBoost,0.847877,0.000000,0.000000,0.00000,0.679507,standard_scaled,none
2,XGBoost,0.658019,0.239482,0.573643,0.33790,0.675163,standard_scaled,scale_pos_weight
3,LightGBM,0.695755,0.238866,0.457364,0.31383,0.643400,standard_scaled,scale_pos_weight


In [5]:
traditional_comparison_df = traditional_comparison_df.rename(
    columns={
        "dataset_version": "Dataset_version",
        "model_name": "Model",
        "imbalance_method": "Imbalance_method",
        "test_accuracy": "Accuracy",
        "test_precision": "Precision",
        "test_recall": "Recall",
        "test_f1": "F1-Score",
        "test_roc_auc": "ROC-AUC"
    }
)

traditional_relevant = traditional_comparison_df[["Dataset_version", "Model", "Imbalance_method", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]
boosting_comparison_df = boosting_comparison_df[["Dataset_version", "Model", "Imbalance_method", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]

In [6]:
comparison_models = pd.concat([traditional_relevant, boosting_comparison_df], ignore_index=True)
display(comparison_models.sort_values("ROC-AUC", ascending=False).head(30))

,Dataset_version,Model,Imbalance_method,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,standard_scaled,svm,class_weight,0.669811,0.252459,0.596899,0.354839,0.689712
1,unscaled,svm,class_weight,0.668632,0.251634,0.596899,0.354023,0.689712
2,robust_scaled,svm,class_weight,0.668632,0.251634,0.596899,0.354023,0.689588
7,robust_scaled,svm,smote,0.650943,0.244648,0.620155,0.350877,0.688887
8,robust_scaled,svm,smote+class_weight,0.650943,0.244648,0.620155,0.350877,0.688887
5,standard_scaled,svm,smote,0.650943,0.244648,0.620155,0.350877,0.687637
6,standard_scaled,svm,smote+class_weight,0.650943,0.244648,0.620155,0.350877,0.687637
3,unscaled,svm,smote,0.650943,0.244648,0.620155,0.350877,0.687615
4,unscaled,svm,smote+class_weight,0.650943,0.244648,0.620155,0.350877,0.687615
49,robust_scaled,logistic_regression,none,0.846698,0.454545,0.038760,0.071429,0.687141


## Model Selection:

NameError: name 'all_results' is not defined